# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access metadata fields
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")
print(f"Dataset DOI/Identifier: {metadata_json['identifier']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We print the available RecordSets and Fields, referencing all by their `@id` fields, as required.

In [ ]:
# List record sets with their @id and label
print("--- Record Sets ---")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"@id: {rs.id} | Name: {rs.name}")
    record_sets.append(rs.id)

if record_sets:
    # For each record set, list the fields by @id and their column ids
    for rs in dataset.metadata.record_sets:
        print(f"\nRecord Set @id: {rs.id}  ({rs.name})")
        if rs.fields:
            print("  Fields:")
            for field in rs.fields:
                if hasattr(field, 'column_ids'):
                    print(f"    @id: {field.id} | Name: {field.name} | column_ids: {field.column_ids}")
                else:
                    print(f"    @id: {field.id} | Name: {field.name}")
        else:
            print("  No fields detected.")
else:
    print("No record sets detected in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

*All entities (record sets, fields, columns) are referenced by their `@id`.*

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Extract all record sets to DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet: {record_set_id}")
    else:
        print(f"No records found for RecordSet: {record_set_id}")

# Pick the *first* record set as the main for subsequent exploration, or choose specifically if known
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in {main_rs_id}: {list(dataframes[main_rs_id].columns)}")
    display(dataframes[main_rs_id].head())
else:
    print('No record sets available to load.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. All field/column references use their Croissant `@id`.

In [ ]:
# Choose a numeric field for filtering and normalization
# We programmatically search for numeric/float/integer columns in the main record set.
df = dataframes[main_rs_id]

# Try to detect numeric fields
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_fields:
    # Try to convert potentially numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    threshold = df[numeric_field_id].mean()  # set threshold at mean for demo
    # Filtering step
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {filtered_df.shape[0]} rows")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try to group by another field (preferably categorical)
    other_fields = [col for col in df.columns if col != numeric_field_id]
    group_field = None
    for col in other_fields:
        if df[col].dtype == object and df[col].nunique() < 10:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
    else:
        print('No suitable categorical field found for grouping.')
else:
    print('No numeric field found for analysis in the dataframe.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize the distribution of the main numeric field
if numeric_fields:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=25)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field_id} in {main_rs_id}')
    plt.show()
    # Boxplot for filtered values
    plt.figure(figsize=(8, 4))
    filtered_df[[numeric_field_id]].boxplot()
    plt.title(f'Boxplot of {numeric_field_id} (> mean)')
    plt.show()
    # If group_field found, plot means per group
    if group_field:
        grouped_df[numeric_field_id].plot(kind='bar', figsize=(8,4))
        plt.ylabel('Mean Value')
        plt.title(f'Mean {numeric_field_id} by {group_field}')
        plt.show()
else:
    print('No numeric fields available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


In this notebook, we successfully loaded and explored the FAIR² dataset of mass spectrometry analysis from Croissant.

- The data record sets, fields, and columns were referenced by their Croissant `@id`s for clarity and reproducibility.
- Basic exploratory analysis and visualizations revealed the distribution and structure of key numeric attributes in the main record set.

Further domain-specific analyses, such as protein annotation and pathway enrichment, can be performed using the detailed field and record set `@id` mappings provided in this dataset.